# F3 · A quantum layer in a neural network: what does one qubit contribute?
**Series F: Quantum Meets Machine Learning, Post 3 of 4** &nbsp;|&nbsp; companion *primer* notebook

The full tutorial of this post (*Hybrid quantum-classical neural networks with PyTorch and Qiskit*, Prof. V. S. Batista, CHEM 584) puts a **one-qubit circuit** at the end of a small convolutional network and trains the whole chain on MNIST zeros and ones with PyTorch and Qiskit. This primer rebuilds every mechanism of that tutorial **from scratch, with NumPy only**, and then asks the question that the tutorial ends with: *does the quantum layer do anything that its formula does not?*

> **Two parts.** **Part I** (up to the self-check) needs only NumPy, pandas and matplotlib and runs in about a minute on a CPU: the circuit as a state vector, shot noise, the parameter-shift rule and the variance of its estimate, a small hybrid network trained with *sampled* circuit outputs and *shifted-circuit* gradients, the effect of the number of shots, and a comparison over ten random seeds. **Part II** installs Qiskit and runs the *same circuit* with Qiskit's sampler, to check the NumPy simulator against it.
>
> **Data.** To stay offline and fast, Part I uses the $8\times8$ digits that ship with scikit-learn (zeros against ones) in place of MNIST. The images are smaller and the classical part is a small perceptron instead of a convolutional network, so the numbers differ from the tutorial's; the *mechanisms* are the same.

| Section | What you do |
|---|---|
| 1 | the circuit as a state vector: $q(\theta)=\tfrac12(1+\sin\theta)$ |
| 2 | shots: the noise of the estimate, and where it is largest |
| 3 | the parameter-shift rule: exact, the factor $1/(2\sin s)$, and the noise of the gradient |
| 4 | a hybrid network, trained with shots and shifted circuits |
| 5 | testing: the number of shots, and two artifacts of the shots table |
| 6 | is the quantum layer doing anything? ten seeds, and a sigmoid for comparison |
| 7 | what training costs in shots |
| **Part II** | run the same circuit with Qiskit |

## 0. Setup

In [ ]:
import math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
q_exact = lambda t: 0.5 * (1 + np.sin(t))            # the probability of measuring 1 (derived in Section 1)

## 1. The circuit by hand
One qubit, $|0\rangle\to H\to R_Y(\theta)\to$ measure. As a vector of two amplitudes, $H=\tfrac1{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ and $R_Y(\theta)=\begin{pmatrix}\cos\frac\theta2&-\sin\frac\theta2\\\sin\frac\theta2&\cos\frac\theta2\end{pmatrix}$. The probability of measuring 1 is the squared modulus of the second amplitude.
**Stop and predict:** what is $P(1)$ for $\theta=\pi$, and for $\theta=\pi/2$?

In [ ]:
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
RY = lambda t: np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]])
Z = np.diag([1.0, -1.0])
def circuit_state(theta): return RY(theta) @ H @ np.array([1.0, 0.0])               # |0> -> H -> RY(theta)

rows = []
for t in (0.0, np.pi / 2, np.pi, 3 * np.pi / 2, 1.0):
    psi = circuit_state(t); rows.append((f"{t:.4f}", psi[1] ** 2, q_exact(t), psi @ Z @ psi, -np.sin(t)))
print(pd.DataFrame(rows, columns=["theta", "P(1) from the state vector", "(1 + sin t)/2", "<Z>", "-sin t"]).round(6).to_string(index=False))
grid = np.linspace(0, 2 * np.pi, 200); err = max(abs(circuit_state(t)[1] ** 2 - q_exact(t)) for t in grid)
print(f"\nlargest difference between the state-vector probability and (1 + sin t)/2 over 200 angles: {err:.1e}")
assert err < 1e-12

## 2. Shots: the noise of the estimate
A quantum computer does not hand over $q$. It runs the circuit `shots` times and returns the measured bits; the layer's output is the **fraction of ones**. With $N$ shots this is a binomial estimate with mean $q$ and variance $q(1-q)/N$. Our simulator draws that binomial directly. Note where the variance is large: at $q=\tfrac12$ (angles $0$ and $\pi$), and zero at $q=0$ or $1$ (angle $\pm\pi/2$).
**Stop and predict:** the standard deviation of the estimate at $\theta=0$ with 100 shots; and at $\theta=\pi/2$.

In [ ]:
rng = np.random.default_rng(0)
def sample_q(theta, shots, rng):
    """What a sampler returns for this circuit: the fraction of the shots in which the qubit was found in |1>."""
    return rng.binomial(shots, q_exact(np.asarray(theta, dtype=float))) / shots

print(f"theta = pi with 100 shots: {sample_q(np.pi, 100, rng):.2f}   (exact value 0.50)")
rows = []
for theta, name in ((0.0, "0 (q = 1/2)"), (np.pi / 4, "pi/4"), (np.pi / 2, "pi/2 (q = 1)")):
    for N in (10, 100, 1000, 10000):
        est = sample_q(np.full(5000, theta), N, rng); th = math.sqrt(q_exact(theta) * (1 - q_exact(theta)) / N)
        rows.append((name, N, est.std(), th))
tab = pd.DataFrame(rows, columns=["theta", "shots", "std (5,000 repeats)", "sqrt(q(1-q)/N)"])
print(tab.round(4).to_string(index=False))
r = tab[(tab.theta == "0 (q = 1/2)") & (tab.shots == 1000)].iloc[0]; assert abs(r.iloc[2] / r.iloc[3] - 1) < 0.05
print("\nto halve the noise one needs four times as many shots; and where q is 0 or 1 there is no noise at all")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); tt = np.linspace(0, 2 * np.pi, 400); N = 100
ax[0].plot(tt, q_exact(tt), "k-", lw=1.5, label=r"exact $\frac{1}{2}(1+\sin\theta)$")
ax[0].fill_between(tt, q_exact(tt) - np.sqrt(q_exact(tt) * (1 - q_exact(tt)) / N), q_exact(tt) + np.sqrt(q_exact(tt) * (1 - q_exact(tt)) / N), color=COLORS["blue"], alpha=0.25, label=r"$\pm1$ std, 100 shots")
th60 = np.linspace(0, 2 * np.pi, 61); ax[0].plot(th60, sample_q(th60, 100, rng), "o", ms=3.5, color=COLORS["blue"], label="one estimate per angle")
ax[0].set(xlabel=r"angle $\theta$", ylabel="P(measure 1)", title="(a) The layer output and its noise"); ax[0].legend(fontsize=8)
ax[1].plot(tt, np.abs(np.cos(tt)) / (2 * np.sqrt(N)), color=COLORS["blue"], label=r"std of the output, $|\cos\theta|/(2\sqrt{N})$")
ax[1].set(xlabel=r"angle $\theta$", ylabel="standard deviation (100 shots)", title="(b) Where the noise is largest"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. The parameter-shift rule
Training needs $\mathrm dq/\mathrm d\theta$, which a real device cannot reveal. For a gate generated by a Pauli matrix, **evaluating the circuit at two shifted angles gives the exact derivative**: $\dfrac{\mathrm dq}{\mathrm d\theta}=\dfrac{q(\theta+s)-q(\theta-s)}{2\sin s}$ for any shift $s$ with $\sin s\ne0$ (the usual choice is $s=\pi/2$, which makes the denominator 2). The factor $1/(2\sin s)$ matters: without it the difference is $2\sin s$ times the derivative.
**Stop and predict:** the *plain difference* $q(\theta+\pi/2)-q(\theta-\pi/2)$, by what factor does it differ from the true derivative?

In [ ]:
theta0 = 0.7; exact_d = 0.5 * np.cos(theta0); rows = []
for s in (np.pi / 2, np.pi / 4, 1.0, 2.0):
    diff = q_exact(theta0 + s) - q_exact(theta0 - s)
    rows.append((f"{s:.4f}", diff / (2 * np.sin(s)), exact_d, diff / exact_d, 2 * np.sin(s)))
print(pd.DataFrame(rows, columns=["shift s", "(q+ - q-)/(2 sin s)", "exact dq/dtheta", "plain difference / exact", "2 sin s"]).round(6).to_string(index=False))
assert all(abs(r[1] - r[2]) < 1e-12 for r in rows) and abs(rows[0][3] - 2) < 1e-12
print("\nthe rule is exact for every shift (it is not a finite difference); at s = pi/2 the plain difference is twice the derivative")

**The noise of the gradient.** With sampled circuits, each of the two estimates has variance $q_\pm(1-q_\pm)/N$ and the estimator has variance $\dfrac{q_+(1-q_+)+q_-(1-q_-)}{4\sin^2 s\,N}$. For $s=\pi/2$ the two probabilities are $\tfrac12(1\pm\cos\theta)$ and this is $\boxed{\sin^2\theta/(8N)}$ (derivation in the post's Supplement). So the gradient is noise-free at $\theta=0$ (where the *output* is noisiest) and is pure noise at $\theta=\pi/2$, where the true gradient is zero and the output is noise-free: the two noises live at different angles.

In [ ]:
N = 100; rows = []
for theta in (0.0, 0.7, 1.2, np.pi / 2, 2.0):
    plus = sample_q(np.full(20000, theta + np.pi / 2), N, rng); minus = sample_q(np.full(20000, theta - np.pi / 2), N, rng)
    est = (plus - minus) / 2
    rows.append((f"{theta:.3f}", est.mean(), 0.5 * np.cos(theta), est.std(), abs(np.sin(theta)) / math.sqrt(8 * N)))
print(pd.DataFrame(rows, columns=["theta", "mean of the estimate", "exact 0.5 cos t", "std of the estimate", "|sin t| / sqrt(8N)"]).round(4).to_string(index=False))
for r in rows[1:3] + rows[4:]: assert abs(r[3] / r[4] - 1) < 0.05
snr = lambda t, N: math.sqrt(2 * N) * abs(1 / np.tan(t)); print("\nsignal-to-noise ratio of one gradient estimate, sqrt(2N)|cot t|, for N = 100: at t = 0.7:", round(snr(0.7, 100), 1), "| at t = 1.5:", round(snr(1.5, 100), 2), "| at pi/2: 0")
# the shift that minimizes the variance
ss = np.linspace(0.05, np.pi - 0.05, 600); var_s = lambda t, s: (q_exact(t + s) * (1 - q_exact(t + s)) + q_exact(t - s) * (1 - q_exact(t - s))) / (4 * np.sin(s) ** 2 * N)
print("variance-minimizing shift at theta = 0.7:", round(float(ss[np.argmin(var_s(0.7, ss))]), 3), "(pi/2 = 1.571); at 2.0:", round(float(ss[np.argmin(var_s(2.0, ss))]), 3))
assert abs(ss[np.argmin(var_s(0.7, ss))] - np.pi / 2) < 0.05

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.9)); tt = np.linspace(0, np.pi, 300); N = 100
ax.plot(tt, np.abs(0.5 * np.cos(tt)), "k-", lw=1.6, label=r"true gradient $|\frac{1}{2}\cos\theta|$")
ax.plot(tt, np.abs(np.sin(tt)) / np.sqrt(8 * N), color=COLORS["red"], label=r"noise of the gradient estimate, $|\sin\theta|/\sqrt{8N}$")
ax.plot(tt, np.abs(np.cos(tt)) / (2 * np.sqrt(N)), color=COLORS["blue"], ls="--", label=r"noise of the layer output, $|\cos\theta|/(2\sqrt{N})$")
ax.set(xlabel=r"angle $\theta$", ylabel="magnitude (100 shots)", title="Signal and noise of the quantum layer"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. A hybrid network, trained with shots
**The model.** The classical part condenses the image to **one number**, the angle $\theta$; the quantum layer turns it into $q=P(1)$; the network outputs the probabilities $(1-q,\,q)$ of the digits 0 and 1 and is trained with the negative log-likelihood $-\log(p_{\text{true}}+\varepsilon)$ ($\varepsilon=10^{-3}$, because a sampled estimate can be exactly 0). The classical part here is a small perceptron, $64\to16\to1$, written with NumPy and trained by Adam. **Three heads**, same classical part and same initial weights: `exact` (the formula $\tfrac12(1+\sin\theta)$ and its exact derivative), `shots` (the sampled circuit and the shifted-circuit gradient, 100 shots each) and, for Section 6, `sigmoid` (an ordinary activation).
The backward pass is the chain rule: $\mathrm dL/\mathrm d\theta=(\mathrm dL/\mathrm dq)\,(\mathrm dq/\mathrm d\theta)$, with $\mathrm dq/\mathrm d\theta$ from the exact formula or from the two shifted circuits.

In [ ]:
from sklearn.datasets import load_digits
d = load_digits(); X = d.data / 16.0; y = d.target; i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
tr = np.r_[i0[:100], i1[:100]]; te = np.r_[i0[100:150], i1[100:150]]                 # 200 training and 100 test images, as in the tutorial
Xtr, ytr, Xte, yte = X[tr], (y[tr] == 1).astype(int), X[te], (y[te] == 1).astype(int)
print("training images", Xtr.shape, "class counts", np.bincount(ytr), "| test images", Xte.shape, "class counts", np.bincount(yte))
EPS = 1e-3

def init(seed, h=16):
    r = np.random.default_rng(seed); return {"W1": r.normal(0, np.sqrt(2 / 64), (64, h)), "b1": np.zeros(h), "W2": r.normal(0, np.sqrt(1 / h), (h, 1)), "b2": np.zeros(1)}
def forward(p, x): z = x @ p["W1"] + p["b1"]; a = np.maximum(z, 0); return z, a, (a @ p["W2"] + p["b2"])[0]

def train_hybrid(seed, head="shots", shots=100, epochs=20, lr=1e-3):
    p = init(seed); rs = np.random.default_rng(seed + 1000); order = np.random.default_rng(seed + 2000)
    m = {k: np.zeros_like(v) for k, v in p.items()}; v = {k: np.zeros_like(v) for k, v in p.items()}; t = 0; hist = []
    for ep in range(epochs):
        L = C = 0
        for i in order.permutation(len(Xtr)):                                          # batch size 1, as in the tutorial's default
            x, yy = Xtr[i], ytr[i]; z, a, theta = forward(p, x)
            if head == "sigmoid": q = 1 / (1 + np.exp(-theta)); dq = q * (1 - q)
            elif head == "exact": q = q_exact(theta); dq = 0.5 * np.cos(theta)
            else:
                q = rs.binomial(shots, q_exact(theta)) / shots                         # forward pass: run the circuit
                qp = rs.binomial(shots, q_exact(theta + np.pi / 2)) / shots; qm = rs.binomial(shots, q_exact(theta - np.pi / 2)) / shots
                dq = (qp - qm) / (2 * np.sin(np.pi / 2))                               # backward pass: the two shifted circuits
            pt = q if yy == 1 else 1 - q; L += -np.log(pt + EPS); C += int((q > 0.5) == bool(yy))
            dth = (-1 / (q + EPS) if yy == 1 else 1 / (1 - q + EPS)) * dq
            da = p["W2"][:, 0] * dth; dz = da * (z > 0); g = {"W2": (a * dth)[:, None], "b2": np.array([dth]), "W1": np.outer(x, dz), "b1": dz}
            t += 1
            for k in p:
                m[k] = 0.9 * m[k] + 0.1 * g[k]; v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
                p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        hist.append((L / len(Xtr), 100 * C / len(Xtr)))
    return p, hist

t0 = time.perf_counter(); p_q, h_q = train_hybrid(0, "shots"); t_q = time.perf_counter() - t0
t0 = time.perf_counter(); p_e, h_e = train_hybrid(0, "exact"); t_e = time.perf_counter() - t0
print(pd.DataFrame({"epoch": [1, 2, 3, 5, 10, 20], "loss (100 shots)": [h_q[e - 1][0] for e in (1, 2, 3, 5, 10, 20)], "train acc (%)": [h_q[e - 1][1] for e in (1, 2, 3, 5, 10, 20)],
                    "loss (exact)": [h_e[e - 1][0] for e in (1, 2, 3, 5, 10, 20)], "train acc (%) exact": [h_e[e - 1][1] for e in (1, 2, 3, 5, 10, 20)]}).round(4).to_string(index=False))
print(f"\ntraining time here: {t_q:.2f} s (shots) and {t_e:.2f} s (exact): a NumPy sampler costs nothing; with a circuit simulator the cost is real (Part II)")
assert h_q[-1][1] >= 99 and h_e[-1][1] >= 99

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6)); ep = np.arange(1, 21)
ax[0].plot(ep, [h[0] for h in h_q], "o-", ms=3.5, color=COLORS["blue"], label="sampled circuit (100 shots)"); ax[0].plot(ep, [h[0] for h in h_e], "s-", ms=3.5, color=COLORS["orange"], label="exact formula")
ax[0].set(xlabel="epoch", ylabel="training loss", title="(a) Training loss"); ax[0].legend(fontsize=8)
ax[1].plot(ep, [h[1] for h in h_q], "o-", ms=3.5, color=COLORS["blue"]); ax[1].plot(ep, [h[1] for h in h_e], "s-", ms=3.5, color=COLORS["orange"])
ax[1].set(xlabel="epoch", ylabel="training accuracy (%)", title="(b) Training accuracy", ylim=(80, 101)); plt.tight_layout(); plt.show()

## 5. Testing, and two artifacts of the shots table
We evaluate the trained model on the 100 test images with a different number of shots at *test time*, and with the exact formula. For each setting we report the accuracy, the loss and the **mean probability assigned to the correct digit**. **Stop and predict:** with one shot, the estimate $\hat q$ is exactly 0 or 1. What does that make of the "mean probability of the correct digit"?

In [ ]:
def evaluate(p, head="exact", shots=100, rng=None):
    th = np.array([forward(p, x)[2] for x in Xte]); qe = 1 / (1 + np.exp(-th)) if head == "sigmoid" else q_exact(th)
    q = qe if head in ("exact", "sigmoid") else rng.binomial(shots, qe) / shots
    pt = np.where(yte == 1, q, 1 - q); return dict(acc=100 * np.mean((q > 0.5) == (yte == 1)), loss=float(np.mean(-np.log(pt + EPS))), pt=float(pt.mean()), theta=th, q_exact=qe, q=q)

rs = np.random.default_rng(5); rows = []; by_shots = {}
for shots in (1, 10, 100, 1000, 10000):
    r = evaluate(p_q, "shots", shots, rs); by_shots[shots] = r; rows.append((str(shots), r["acc"], r["loss"], r["pt"]))
r = evaluate(p_q, "exact"); rows.append(("exact", r["acc"], r["loss"], r["pt"])); exact_pt = r["pt"]
print(pd.DataFrame(rows, columns=["shots at test time", "accuracy (%)", "loss", "mean prob. of the true digit"]).round(3).to_string(index=False))
one = by_shots[1]; assert abs(one["acc"] / 100 - one["pt"]) < 1e-12
print(f"\nartifact 1: with ONE shot the 'probability of the true digit' is 1 if the estimate is right and 0 if wrong, so it equals the accuracy ({one['acc']:.1f} % and {one['pt']:.3f}); it says nothing about confidence")
exp1 = float(np.mean([evaluate(p_q, "shots", 1, rs)["acc"] for _ in range(300)]))
print(f"the expected one-shot accuracy is the exact mean probability of the true digit: {100 * exact_pt:.2f} % predicted, {exp1:.2f} % measured over 300 repeats")
assert abs(exp1 - 100 * exact_pt) < 0.6
w = 1 - one["acc"] / 100; formula = (1 - w) * (-math.log(1 + EPS)) + w * (-math.log(EPS))
print(f"artifact 2: with one shot every estimate is 0 or 1, so the loss is (1 - w)(-log(1 + eps)) + w(-log eps), with w the error rate: {formula:.3f} (measured {one['loss']:.3f}).")
print(f"           With no error it is -log(1.001) = {-math.log(1 + EPS):.4f}: a NEGATIVE loss (the 1-shot row of the tutorial's table); every error costs {-math.log(EPS):.1f}")
assert abs(formula - one["loss"]) < 1e-9

In [ ]:
r = evaluate(p_q, "exact"); th, qe = r["theta"], r["q_exact"]
sat = float(np.mean((qe < 0.05) | (qe > 0.95)))
print(f"test angles: min {th.min():.2f}, median {np.median(th):.2f}, max {th.max():.2f} (the output peaks at +pi/2 = 1.57 and bottoms at -pi/2)")
print(f"share of the test images with q < 0.05 or q > 0.95: {100 * sat:.0f} %")
print(f"average noise of the output over the test images (100 shots): {np.mean(np.sqrt(qe * (1 - qe) / 100)):.4f}  against 0.05 at q = 1/2")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(th[yte == 0], bins=20, range=(-2.2, 2.2), color=COLORS["orange"], alpha=0.8, label="digit 0"); ax[0].hist(th[yte == 1], bins=20, range=(-2.2, 2.2), color=COLORS["blue"], alpha=0.8, label="digit 1")
ax[0].axvline(np.pi / 2, color="k", ls=":"); ax[0].axvline(-np.pi / 2, color="k", ls=":"); ax[0].set(xlabel=r"angle $\theta$ on the test images", ylabel="images", title="(a) The network pushes the angle towards the extremes"); ax[0].legend(fontsize=8)
tt = np.linspace(-2.2, 2.2, 200); ax[1].plot(tt, q_exact(tt), "k-"); ax[1].plot(th[yte == 0], qe[yte == 0], "o", ms=3, color=COLORS["orange"]); ax[1].plot(th[yte == 1], qe[yte == 1], "o", ms=3, color=COLORS["blue"])
ax[1].set(xlabel=r"angle $\theta$", ylabel=r"$q=P(1)$", title="(b) ... where q is nearly 0 or 1 and the shot noise vanishes"); plt.tight_layout(); plt.show()
assert sat > 0.7

## 6. Is the quantum layer doing anything?
In Section 2 we showed that the layer computes $q(\theta)=\tfrac12(1+\sin\theta)$ plus shot noise. The `exact` head computes that function directly. If the two behave alike, the circuit has contributed nothing that the formula does not. A single run is not a statistical test, so we repeat the comparison for **ten seeds**, add the `sigmoid` head as a third network, and compare the test results (100 shots at test time for the sampled layer).

In [ ]:
rows = []
for sd in range(10):
    pq, _ = train_hybrid(sd, "shots"); pe, _ = train_hybrid(sd, "exact"); ps, _ = train_hybrid(sd, "sigmoid")
    rq, re_, rg = evaluate(pq, "shots", 100, np.random.default_rng(sd)), evaluate(pe, "exact"), evaluate(ps, "sigmoid")
    rows.append(dict(seed=sd, acc_quantum=rq["acc"], loss_quantum=rq["loss"], acc_exact=re_["acc"], loss_exact=re_["loss"], acc_sigmoid=rg["acc"], loss_sigmoid=rg["loss"]))
df = pd.DataFrame(rows); print(df.round(3).to_string(index=False))
summary = df.drop(columns="seed").agg(["mean", "std"]); print("\n" + summary.round(3).to_string())
dl = df.loss_quantum - df.loss_exact; t_stat = dl.mean() / (dl.std(ddof=1) / math.sqrt(len(dl)))
print(f"\npaired difference of the test loss, sampled layer minus exact formula: {dl.mean():+.4f} +- {dl.std(ddof=1):.4f} (t = {t_stat:.2f} with 9 degrees of freedom; |t| > 2.26 would be significant at 5 %)")
assert summary.loc["mean", "acc_quantum"] >= 99 and summary.loc["mean", "acc_exact"] >= 99 and abs(t_stat) < 2.5

**Reading the table.** The sampled quantum layer and its exact formula are statistically indistinguishable in accuracy and loss: the circuit adds shot noise and nothing else. The sigmoid, a plain activation, does as well in accuracy and *better in loss*. The reason is the shape of $\tfrac12(1+\sin\theta)$: it is **bounded and flat at its extremes** (near $\theta=\pi/2$, $q\approx1-\tfrac14(\theta-\pi/2)^2$ and the gradient vanishes), so the network can approach a probability of 1 only slowly, whereas a sigmoid keeps increasing. That is a property of this particular one-qubit function, not of quantum mechanics in general.

## 7. What training costs in shots
Every training step evaluates the circuit once forward and twice backward (the two shifts). With $S$ steps and $N$ shots per evaluation the quantum computer is asked for $3SN$ shots. Here $S=200\times20=4{,}000$ steps and $N=100$.

In [ ]:
steps, N = 200 * 20, 100
shots_total = 3 * steps * N
print(f"circuit evaluations: {3 * steps:,} | shots: {shots_total:,}  ({shots_total / 1e6:.1f} million) for ONE training run of 20 epochs")
print(f"with 1,000 shots per evaluation: {3 * steps * 1000 / 1e6:.0f} million shots; with the gradient noise at q ~ 1 of order 1/sqrt(8N) = {1 / math.sqrt(8 * N):.3f}")

## 8. Your turn
**Exercise 1.** For which angle is the plain-difference gradient (without the factor $1/(2\sin s)$) correct? Is that a useful angle?
**Exercise 2.** Change $R_Y$ to $R_X$ in Section 1. What is the new $q(\theta)$ (use the state vector), and does the parameter-shift rule still give its derivative with the same shift?
**Exercise 3.** Train with 10 and 1,000 shots (`train_hybrid(0, "shots", shots=...)`). How do the curves and the test accuracy change?
**Exercise 4.** Add readout noise: flip each measured bit with probability 5 %. How does the estimate of $q$ change, and how would you correct it?
**Exercise 5.** Use a batch of 8 images in `train_hybrid` (average the gradients). What happens to the noise of the update?

## 9. Self-check

In [ ]:
assert err < 1e-12
assert all(abs(r[1] - r[2]) < 1e-12 for r in [(None, (q_exact(0.7 + s) - q_exact(0.7 - s)) / (2 * np.sin(s)), 0.5 * np.cos(0.7)) for s in (np.pi / 2, 1.0, 2.0)])
assert abs(one["acc"] / 100 - one["pt"]) < 1e-12 and abs(exp1 - 100 * exact_pt) < 0.6
assert summary.loc["mean", "acc_quantum"] >= 99 and abs(t_stat) < 2.5 and sat > 0.7
print("All checks passed: you reproduced the numbers in the post.")

# Part II. The same circuit in Qiskit
> **Read this first.** Part II installs **Qiskit** (`pip install qiskit pylatexenc`) and runs the circuit of Section 1 with Qiskit's `StatevectorSampler`, the reference simulator of the *primitives* interface: the sampler receives `(circuit, angles)` and returns the measured bit strings, exactly as a device would. The cells check the NumPy simulator of Part I against it: the estimates, the shot noise and the parameter-shift gradient. The full tutorial (Notebook 1 of the post) goes on to train the hybrid CNN on MNIST with PyTorch.

In [ ]:
!pip install -q qiskit pylatexenc

In [ ]:
HAVE_QISKIT = importlib.util.find_spec("qiskit") is not None
if HAVE_QISKIT:
    import qiskit
    from qiskit import QuantumCircuit
    from qiskit.circuit import Parameter
    from qiskit.primitives import StatevectorSampler
    theta_p = Parameter("theta"); qc = QuantumCircuit(1); qc.h(0); qc.barrier(); qc.ry(theta_p, 0); qc.measure_all()
    print("qiskit", qiskit.__version__); print(qc.draw("text"))
    def qiskit_q(thetas, shots, seed=0):
        """The estimate of P(1) for each angle: the fraction of ones among the shots."""
        thetas = np.asarray(thetas, dtype=float).reshape(-1, 1)
        res = StatevectorSampler(default_shots=shots, seed=seed).run([(qc, thetas)]).result()[0]
        bits = res.data.meas
        return np.array([bits.get_counts(i).get("1", 0) for i in range(len(thetas))]) / shots
else:
    print("Qiskit is not available: run the install cell above (restart the session if Colab asks), then re-run this cell.")

In [ ]:
# 1) the estimates against the exact formula, at many shots
if HAVE_QISKIT:
    tt = np.linspace(0, 2 * np.pi, 41); est = qiskit_q(tt, 50_000)
    z = (est - q_exact(tt)) / np.sqrt(np.maximum(q_exact(tt) * (1 - q_exact(tt)), 1e-12) / 50_000)
    print(f"50,000 shots: largest |estimate - exact| = {np.abs(est - q_exact(tt)).max():.4f} (the binomial prediction of the largest deviation of 41 points is about 0.005)")
    assert np.abs(est - q_exact(tt)).max() < 0.01

In [ ]:
# 2) shot noise from Qiskit: 200 repeated estimates at theta = 0 inside ONE call, with a seeded and with an unseeded sampler
if HAVE_QISKIT:
    rows = []
    for N in (10, 100, 1000):
        seeded = qiskit_q(np.zeros(200), N, seed=0); fresh = qiskit_q(np.zeros(200), N, seed=None); ns = sample_q(np.zeros(200), N, np.random.default_rng(1))
        rows.append((N, seeded.std(), fresh.std(), ns.std(), 0.5 / math.sqrt(N)))
    print(pd.DataFrame(rows, columns=["shots", "Qiskit std, seed=0", "Qiskit std, seed=None", "NumPy sampler std", "theory 1/(2 sqrt N)"]).round(4).to_string(index=False))
    angles = np.linspace(0, 6, 9)
    a, b = qiskit_q(angles, 100, seed=0), qiskit_q(angles, 100, seed=0); c, e = qiskit_q(angles, 100, seed=None), qiskit_q(angles, 100, seed=None)
    print("two calls with seed=0 give identical estimates:", np.array_equal(a, b), "| two calls with seed=None:", np.array_equal(c, e))
    assert all(abs(r[2] / r[4] - 1) < 0.2 for r in rows)                # an unseeded sampler behaves like the binomial of Section 2
    if max(r[1] for r in rows) < 0.2 * rows[0][4]:
        print("\nthe seed=0 column is (almost) zero: with a seed, the 200 copies of the same angle in one call gave the SAME estimate, so their shots are not independent;")
        print("the unseeded sampler follows the binomial formula. A fixed seed is convenient for reproducibility but changes what 'shot noise' means in a training loop.")

In [ ]:
# 3) the parameter-shift gradient from Qiskit, with the normalization 1/(2 sin s), and the plain difference.
#    Two ways to call the sampler: independent random numbers for the two shifted circuits, or the same seed in both calls
#    (the two estimates may then share their random numbers; and without a seed they are independent and fresh)
if HAVE_QISKIT:
    tt = np.linspace(0, 2 * np.pi, 41); s = np.pi / 2; exact_g = 0.5 * np.cos(tt); big = np.abs(exact_g) > 0.2
    for name, (seed_plus, seed_minus) in (("independent seeds", (1, 2)), ("the same seed in both calls", (1, 1)), ("no seed", (None, None))):
        diff = qiskit_q(tt + s, 50_000, seed=seed_plus) - qiskit_q(tt - s, 50_000, seed=seed_minus); grad = diff / (2 * np.sin(s))
        print(f"{name:30s}: max |parameter-shift - exact| = {np.abs(grad - exact_g).max():.4f} | plain difference / exact derivative = {np.median(diff[big] / exact_g[big]):.2f}")
        assert np.abs(grad - exact_g).max() < 0.01 and abs(np.median(diff[big] / exact_g[big]) - 2) < 0.05
    print("the standard deviation of one shifted-circuit estimate is at most 1/sqrt(8N) = 0.0016 at N = 50,000; the tutorial reports a maximum error of 0.0010 for its run")

### What to look for
* **Agreement.** The estimates scatter around $\tfrac12(1+\sin\theta)$ by what the binomial predicts, and the *unseeded* noise column matches the formula and the NumPy sampler: the sampler of Part I is a faithful stand-in. **Check the seeded column**: if it is zero, a seeded sampler returns the same estimate for every copy of an angle in one call.
* **The gradient.** With the factor $1/(2\sin s)$ the shifted circuits give the derivative to a few $10^{-3}$ at 50,000 shots (smaller if the two calls share their random numbers); the plain difference is twice too large.
* **Cost.** Qiskit simulates the circuit (a $2\times2$ matrix product) and then draws the shots; the full tutorial's hybrid network needed 39 s for 20 epochs, against 11 s with the exact formula, for the same accuracy.